<a href="https://colab.research.google.com/github/mri2004/fake-news-detection-using-ML/blob/main/DML_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: Install all required libraries
!pip install -q transformers datasets
!pip install -q lime shap
!pip install -q xgboost lightgbm
!pip install -q torch torchvision torchaudio
!pip install -q scikit-learn pandas numpy matplotlib seaborn
!pip install -q tqdm

print("✅ All libraries installed successfully!")

In [ ]:
# Cell 2: Import all required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import time
import warnings
from tqdm import tqdm

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc,
    roc_auc_score
)
from sklearn.utils.class_weight import compute_class_weight

# XGBoost
from xgboost import XGBClassifier

# Deep Learning
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW  # Import AdamW from torch.optim
from transformers import (
    BertTokenizer, BertForSequenceClassification,
    get_linear_schedule_with_warmup
)

# Explainable AI
import lime
from lime.lime_text import LimeTextExplainer
import shap

# Suppress warnings
warnings.filterwarnings('ignore')

# Set device for PyTorch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Set random seeds for reproducibility
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
print("✅ All imports complete!")

In [ ]:
# Cell 3: Mount Google Drive and upload dataset files
from google.colab import drive
from google.colab import files
import os

# Mount Google Drive (optional - for saving models)
try:
    drive.mount('/content/drive')
    print("✅ Google Drive mounted")
except:
    print("⚠️ Could not mount Drive (continuing without)")

print("\n📁 Please upload your LIAR dataset files (train.tsv, valid.tsv, test.tsv):")
print("   Note: Upload each file one by one when prompted")

# Dictionary to store uploaded files
uploaded_files = {}

# Upload train.tsv
print("\n" + "="*50)
print("Uploading train.tsv...")
print("="*50)
uploaded = files.upload()
uploaded_files['train'] = list(uploaded.keys())[0]

# Upload valid.tsv
print("\n" + "="*50)
print("Uploading valid.tsv...")
print("="*50)
uploaded = files.upload()
uploaded_files['valid'] = list(uploaded.keys())[0]

# Upload test.tsv
print("\n" + "="*50)
print("Uploading test.tsv...")
print("="*50)
uploaded = files.upload()
uploaded_files['test'] = list(uploaded.keys())[0]

print("\n✅ All files uploaded successfully!")
print(f"   Train file: {uploaded_files['train']}")
print(f"   Valid file: {uploaded_files['valid']}")
print(f"   Test file: {uploaded_files['test']}")

In [ ]:
# Cell 4: Load LIAR dataset using official splits
print("\n" + "="*80)
print("LOADING LIAR DATASET (OFFICIAL SPLITS)")
print("="*80)

# Column names for LIAR dataset
column_names = [
    'id', 'label', 'statement', 'subject', 'speaker', 'job', 'state',
    'party', 'barely_true_counts', 'false_counts', 'half_true_counts',
    'mostly_true_counts', 'pants_fire_counts', 'context', 'date', 'credit'
]

# Function to load and process each split
def load_liar_split(file_path, split_name):
    """Load and process a LIAR dataset split"""
    df = pd.read_csv(file_path, sep='\t', names=column_names, header=None)

    # Map labels to binary
    def map_label(label):
        fake_labels = ['pants-fire', 'false', 'barely-true']
        real_labels = ['half-true', 'mostly-true', 'true']
        if label in fake_labels:
            return 1  # Fake
        elif label in real_labels:
            return 0  # Real
        return -1  # Unknown

    df['binary_label'] = df['label'].apply(map_label)

    # Remove unknown labels
    unknown_count = len(df[df['binary_label'] == -1])
    df = df[df['binary_label'] != -1]

    print(f"   {split_name}: {len(df)} samples (Real: {sum(df['binary_label']==0)}, Fake: {sum(df['binary_label']==1)})")
    if unknown_count > 0:
        print(f"      Removed {unknown_count} unknown samples")

    return df

# Load all three splits
print("\n📁 Loading LIAR dataset splits...")
train_df = load_liar_split(uploaded_files['train'], 'Training')
valid_df = load_liar_split(uploaded_files['valid'], 'Validation')
test_df = load_liar_split(uploaded_files['test'], 'Test')

print(f"\n✅ Total samples: {len(train_df) + len(valid_df) + len(test_df)}")
print(f"   Train: {len(train_df)}")
print(f"   Validation: {len(valid_df)}")
print(f"   Test: {len(test_df)}")

# Display samples
print(f"\n📝 Sample statements from training set:")
for i in range(min(3, len(train_df))):
    print(f"\n   {i+1}. Statement: {train_df['statement'].iloc[i][:100]}...")
    print(f"      Original Label: {train_df['label'].iloc[i]} → Binary: {'FAKE' if train_df['binary_label'].iloc[i] == 1 else 'REAL'}")

In [ ]:
# Cell 5: Visualize dataset distribution
print("\n" + "="*80)
print("DATA VISUALIZATION")
print("="*80)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 1. Original label distribution (Training)
ax1 = axes[0]
label_counts = train_df['label'].value_counts()
colors_label = ['#ff6b6b', '#ff9f6b', '#ffd96b', '#6bcb77', '#4d96ff', '#9b6bff']

bars1 = ax1.bar(range(len(label_counts)), label_counts.values,
                color=colors_label[:len(label_counts)], edgecolor='black')

ax1.set_title('Original LIAR Labels (Training Set)', fontsize=12, fontweight='bold')
ax1.set_xlabel('Label', fontsize=10)
ax1.set_ylabel('Count', fontsize=10)
ax1.set_xticks(range(len(label_counts)))
ax1.set_xticklabels(label_counts.index, rotation=45, ha='right')

for bar, val in zip(bars1, label_counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 10,
             str(val), ha='center', fontsize=9)

# 2. Binary label distribution (All splits)
ax2 = axes[1]
splits = ['Train', 'Validation', 'Test']

real_counts = [
    sum(train_df['binary_label'] == 0),
    sum(valid_df['binary_label'] == 0),
    sum(test_df['binary_label'] == 0)
]

fake_counts = [
    sum(train_df['binary_label'] == 1),
    sum(valid_df['binary_label'] == 1),
    sum(test_df['binary_label'] == 1)
]

x = np.arange(len(splits))
width = 0.35

ax2.bar(x - width/2, real_counts, width,
        label='REAL', edgecolor='black')
ax2.bar(x + width/2, fake_counts, width,
        label='FAKE', edgecolor='black')

ax2.set_xlabel('Split', fontsize=12)
ax2.set_ylabel('Count', fontsize=12)
ax2.set_title('Binary Label Distribution Across Splits', fontsize=12, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(splits)
ax2.legend()

# 3. Class balance pie chart (Training)
ax3 = axes[2]
train_real = sum(train_df['binary_label'] == 0)
train_fake = sum(train_df['binary_label'] == 1)

ax3.pie([train_real, train_fake],
        labels=['REAL', 'FAKE'],
        autopct='%1.1f%%',
        startangle=90,
        explode=(0.05, 0.05))

ax3.set_title(f'Training Set Class Balance\nTotal: {len(train_df)} samples',
              fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('liar_dataset_visualization.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Visualization saved as 'liar_dataset_visualization.png'")

In [ ]:
# Cell 6: Text preprocessing on all splits
print("\n" + "="*80)
print("TEXT PREPROCESSING")
print("="*80)

def clean_text(text):
    """Comprehensive text cleaning function"""
    # Convert to string and lowercase
    text = str(text).lower()

    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)

    # Remove mentions (@user)
    text = re.sub(r'@\w+', '', text)

    # Remove hashtags (keep the word)
    text = re.sub(r'#', '', text)

    # Remove punctuation and digits
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\d+', '', text)

    # Remove extra whitespace
    text = ' '.join(text.split())

    return text

print("Cleaning text for all splits...")
train_df['cleaned_statement'] = train_df['statement'].apply(clean_text)
valid_df['cleaned_statement'] = valid_df['statement'].apply(clean_text)
test_df['cleaned_statement'] = test_df['statement'].apply(clean_text)

print("\n✅ Text cleaning completed!")
print(f"\nDataframe columns now contain 'cleaned_statement':")
print(f"   Train columns: {train_df.columns.tolist()[:5]}...")

print("\n📝 Sample of cleaned text (Training):")
for i in range(min(3, len(train_df))):
    print(f"\n   Original: {train_df['statement'].iloc[i][:100]}...")
    print(f"   Cleaned:  {train_df['cleaned_statement'].iloc[i][:100]}...")

# Store cleaned texts as lists for easier access
train_texts = train_df['cleaned_statement'].tolist()
valid_texts = valid_df['cleaned_statement'].tolist()
test_texts = test_df['cleaned_statement'].tolist()

train_labels = train_df['binary_label'].tolist()
valid_labels = valid_df['binary_label'].tolist()
test_labels = test_df['binary_label'].tolist()

print(f"\n✅ Data prepared:")
print(f"   Training: {len(train_texts)} samples")
print(f"   Validation: {len(valid_texts)} samples")
print(f"   Test: {len(test_texts)} samples")

In [ ]:
# Cell 7: TF-IDF Feature Extraction (fit on TRAIN only!)
print("\n" + "="*80)
print("TF-IDF FEATURE EXTRACTION")
print("="*80)

# Initialize TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    max_df=0.8,
    min_df=2
)

# IMPORTANT: Fit ONLY on training data, then transform all splits
print("Fitting TF-IDF on training data only...")
X_train = tfidf_vectorizer.fit_transform(train_texts)

print("Transforming validation and test data...")
X_val = tfidf_vectorizer.transform(valid_texts)
X_test = tfidf_vectorizer.transform(test_texts)

# Labels
y_train = np.array(train_labels)
y_val = np.array(valid_labels)
y_test = np.array(test_labels)

print(f"\n✅ Feature extraction completed!")
print(f"   Training features: {X_train.shape}")
print(f"   Validation features: {X_val.shape}")
print(f"   Test features: {X_test.shape}")
print(f"   Number of features: {X_train.shape[1]}")
print(f"   Sparsity: {100 * (1 - X_train.nnz / (X_train.shape[0] * X_train.shape[1])):.2f}% zeros")

# Get feature names for later use
feature_names = tfidf_vectorizer.get_feature_names_out()
print(f"\n   First 10 features: {feature_names[:10].tolist()}")

In [ ]:
# Cell 8: Handle class imbalance
print("\n" + "="*80)
print("CLASS IMBALANCE HANDLING")
print("="*80)

# Compute class weights based on training data
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}

print(f"\n⚖️ Class weights:")
print(f"   REAL (0): {class_weight_dict[0]:.4f}")
print(f"   FAKE (1): {class_weight_dict[1]:.4f}")

print(f"\n📊 Training set class distribution:")
print(f"   REAL: {np.sum(y_train == 0)} ({np.sum(y_train == 0)/len(y_train)*100:.1f}%)")
print(f"   FAKE: {np.sum(y_train == 1)} ({np.sum(y_train == 1)/len(y_train)*100:.1f}%)")

print(f"\n📊 Validation set class distribution:")
print(f"   REAL: {np.sum(y_val == 0)} ({np.sum(y_val == 0)/len(y_val)*100:.1f}%)")
print(f"   FAKE: {np.sum(y_val == 1)} ({np.sum(y_val == 1)/len(y_val)*100:.1f}%)")

print(f"\n📊 Test set class distribution:")
print(f"   REAL: {np.sum(y_test == 0)} ({np.sum(y_test == 0)/len(y_test)*100:.1f}%)")
print(f"   FAKE: {np.sum(y_test == 1)} ({np.sum(y_test == 1)/len(y_test)*100:.1f}%)")

In [ ]:
# Cell 9: Train all traditional models
print("\n" + "="*80)
print("TRAINING TRADITIONAL MODELS")
print("="*80)

models = {}
predictions = {}
probabilities = {}
training_times = {}

# 1. Naive Bayes
print("\n1️⃣ Training Naive Bayes Classifier...")
start = time.time()
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)
models['Naive Bayes'] = nb_model
predictions['Naive Bayes'] = nb_model.predict(X_test)
probabilities['Naive Bayes'] = nb_model.predict_proba(X_test)[:, 1]
training_times['Naive Bayes'] = time.time() - start
print(f"   ✅ Completed in {training_times['Naive Bayes']:.2f}s")

# 2. Logistic Regression
print("\n2️⃣ Training Logistic Regression...")
start = time.time()
lr_model = LogisticRegression(random_state=42, max_iter=1000, class_weight=class_weight_dict)
lr_model.fit(X_train, y_train)
models['Logistic Regression'] = lr_model
predictions['Logistic Regression'] = lr_model.predict(X_test)
probabilities['Logistic Regression'] = lr_model.predict_proba(X_test)[:, 1]
training_times['Logistic Regression'] = time.time() - start
print(f"   ✅ Completed in {training_times['Logistic Regression']:.2f}s")

# 3. Random Forest
print("\n3️⃣ Training Random Forest...")
start = time.time()
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight=class_weight_dict, n_jobs=-1)
rf_model.fit(X_train, y_train)
models['Random Forest'] = rf_model
predictions['Random Forest'] = rf_model.predict(X_test)
probabilities['Random Forest'] = rf_model.predict_proba(X_test)[:, 1]
training_times['Random Forest'] = time.time() - start
print(f"   ✅ Completed in {training_times['Random Forest']:.2f}s")

# 4. XGBoost
print("\n4️⃣ Training XGBoost...")
start = time.time()
xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)
xgb_model.fit(X_train, y_train)
models['XGBoost'] = xgb_model
predictions['XGBoost'] = xgb_model.predict(X_test)
probabilities['XGBoost'] = xgb_model.predict_proba(X_test)[:, 1]
training_times['XGBoost'] = time.time() - start
print(f"   ✅ Completed in {training_times['XGBoost']:.2f}s")

print("\n🎉 All traditional models trained successfully!")
print(f"Models: {list(models.keys())}")

In [ ]:
# Cell 10: Evaluate all traditional models
print("\n" + "="*80)
print("TRADITIONAL MODELS EVALUATION (ON TEST SET)")
print("="*80)

results_list = []

for model_name in models.keys():
    accuracy = accuracy_score(y_test, predictions[model_name])
    precision = precision_score(y_test, predictions[model_name])
    recall = recall_score(y_test, predictions[model_name])
    f1 = f1_score(y_test, predictions[model_name])
    auc_score = roc_auc_score(y_test, probabilities[model_name])

    results_list.append({
        'Model': model_name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'AUC': auc_score,
        'Training Time (s)': training_times[model_name]
    })

    print(f"\n📊 {model_name}:")
    print(f"   Accuracy:  {accuracy:.4f}")
    print(f"   Precision: {precision:.4f}")
    print(f"   Recall:    {recall:.4f}")
    print(f"   F1-Score:  {f1:.4f}")
    print(f"   AUC:       {auc_score:.4f}")
    print(f"   Training Time: {training_times[model_name]:.2f}s")

results_df = pd.DataFrame(results_list)
print("\n" + "="*50)
print("SUMMARY - TRADITIONAL MODELS (TEST SET):")
print(results_df.round(4).to_string(index=False))

In [ ]:
# Cell 11: Visualizations for traditional models
print("\n" + "="*80)
print("TRADITIONAL MODELS VISUALIZATIONS")
print("="*80)

fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# 1. Accuracy Comparison
ax1 = axes[0, 0]
bars1 = ax1.bar(results_df['Model'], results_df['Accuracy'],
                color=['#3498db', '#2ecc71', '#e74c3c', '#9b59b6'],
                edgecolor='black')
ax1.set_title('Model Accuracy Comparison (Test Set)', fontsize=14, fontweight='bold')
ax1.set_ylabel('Accuracy', fontsize=12)
ax1.set_ylim(0, 1)
for bar in bars1:
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height + 0.01, f'{height:.3f}',
             ha='center', fontweight='bold')
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45)

# 2. ROC Curves
ax2 = axes[0, 1]
colors_roc = {'Naive Bayes': 'blue', 'Logistic Regression': 'green', 'Random Forest': 'red', 'XGBoost': 'purple'}
for model_name in models.keys():
    fpr, tpr, _ = roc_curve(y_test, probabilities[model_name])
    roc_auc = auc(fpr, tpr)
    ax2.plot(fpr, tpr, color=colors_roc[model_name], lw=2, label=f'{model_name} (AUC={roc_auc:.3f})')
ax2.plot([0, 1], [0, 1], 'gray', lw=2, linestyle='--', label='Random Classifier')
ax2.set_xlabel('False Positive Rate', fontsize=12)
ax2.set_ylabel('True Positive Rate', fontsize=12)
ax2.set_title('ROC Curves - All Models', fontsize=14, fontweight='bold')
ax2.legend(loc='lower right')
ax2.grid(True, alpha=0.3)

# 3. Metrics Heatmap
ax3 = axes[1, 0]
metrics_matrix = results_df[['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC']].values
sns.heatmap(metrics_matrix, annot=True, fmt='.4f', cmap='RdYlGn', ax=ax3,
            xticklabels=['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC'],
            yticklabels=results_df['Model'])
ax3.set_title('Performance Metrics Heatmap', fontsize=14, fontweight='bold')

# 4. Training Time Comparison
ax4 = axes[1, 1]
bars4 = ax4.barh(list(training_times.keys()), list(training_times.values()), color='#3498db', edgecolor='black')
ax4.set_xlabel('Training Time (seconds)', fontsize=12)
ax4.set_title('Model Training Time Comparison', fontsize=14, fontweight='bold')
for bar, time_val in zip(bars4, training_times.values()):
    ax4.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2, f'{time_val:.1f}s', va='center')

plt.tight_layout()
plt.savefig('traditional_models_visualizations.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Visualizations saved as 'traditional_models_visualizations.png'")

In [ ]:
# Confusion Matrices for all models
from sklearn.metrics import confusion_matrix
import seaborn as sns

print("\n" + "="*80)
print("CONFUSION MATRICES")
print("="*80)

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.flatten()

for i, model_name in enumerate(models.keys()):
    y_pred = models[model_name].predict(X_test)
    cm = confusion_matrix(y_test, y_pred)

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['REAL', 'FAKE'],
                yticklabels=['REAL', 'FAKE'])

    axes[i].set_title(f'{model_name} Confusion Matrix', fontweight='bold')
    axes[i].set_xlabel('Predicted Label')
    axes[i].set_ylabel('True Label')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Confusion matrices saved as 'confusion_matrices.png'")

In [ ]:
# Cell: Complete BERT Training + Traditional Models Comparison with Visualizations
print("\n" + "="*80)
print("COMPLETE FAKE NEWS DETECTION: BERT + TRADITIONAL MODELS")
print("="*80)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve, auc,
    classification_report
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
import torch
import warnings
warnings.filterwarnings('ignore')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n🔧 Using device: {device}")

# ============================================================
# STEP 1: Prepare Data for BERT WITH METADATA
# ============================================================
print("\n" + "="*60)
print("STEP 1: PREPARING DATA FOR BERT (WITH METADATA)")
print("="*60)

# Function to combine statement with metadata
def combine_with_metadata(row):
    """Combine statement with speaker, party, and subject information"""

    # Get the statement text
    statement = str(row['statement'])

    # Get metadata (fill missing values with 'unknown')
    speaker = str(row['speaker']) if pd.notna(row['speaker']) else 'unknown'
    party = str(row['party']) if pd.notna(row['party']) else 'unknown'
    subject = str(row['subject']) if pd.notna(row['subject']) else 'unknown'
    job = str(row['job']) if pd.notna(row['job']) else 'unknown'

    # Create enhanced text with metadata
    # Format: [SPEAKER] speaker [PARTY] party [SUBJECT] subject [JOB] job [TEXT] statement
    enhanced_text = f"[SPEAKER] {speaker} [PARTY] {party} [SUBJECT] {subject} [JOB] {job} [TEXT] {statement}"

    return enhanced_text

# Apply metadata combination to training data
print("Adding metadata to training statements...")
train_df['enhanced_text'] = train_df.apply(combine_with_metadata, axis=1)

print("Adding metadata to test statements...")
test_df['enhanced_text'] = test_df.apply(combine_with_metadata, axis=1)

# Now use the enhanced text instead of plain statements
bert_train_texts = train_df['enhanced_text'].tolist()
bert_test_texts = test_df['enhanced_text'].tolist()

# Get labels
if hasattr(y_train, 'values'):
    bert_train_labels = y_train.values.flatten()
else:
    bert_train_labels = np.array(y_train).flatten()

if hasattr(y_test, 'values'):
    bert_test_labels = y_test.values.flatten()
else:
    bert_test_labels = np.array(y_test).flatten()

print(f"\n✅ Enhanced text created with metadata!")
print(f"   Training samples: {len(bert_train_texts)}")
print(f"   Test samples: {len(bert_test_texts)}")

# Show example of enhanced text
print(f"\n📝 Example of enhanced text:")
print(f"   Original: {train_df['statement'].iloc[0][:100]}...")
print(f"   Enhanced: {train_df['enhanced_text'].iloc[0][:150]}...")

# ============================================================
# STEP 2: Tokenize Data for BERT
# ============================================================
print("\n" + "="*60)
print("STEP 2: TOKENIZING DATA FOR BERT")
print("="*60)

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=128
    )

# Create Hugging Face datasets
train_dataset = Dataset.from_dict({'text': bert_train_texts, 'label': bert_train_labels})
test_dataset = Dataset.from_dict({'text': bert_test_texts, 'label': bert_test_labels})

# Tokenize
train_tokenized = train_dataset.map(tokenize_function, batched=True)
test_tokenized = test_dataset.map(tokenize_function, batched=True)

# Remove text column and set format
train_tokenized = train_tokenized.remove_columns(['text'])
test_tokenized = test_tokenized.remove_columns(['text'])
train_tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
test_tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

print(f"✅ Tokenization complete!")
print(f"   Train dataset size: {len(train_tokenized)}")
print(f"   Test dataset size: {len(test_tokenized)}")

# ============================================================
# STEP 3: Load BERT Model
# ============================================================
print("\n" + "="*60)
print("STEP 3: LOADING BERT MODEL")
print("="*60)

bert_model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
bert_model = bert_model.to(device)

total_params = sum(p.numel() for p in bert_model.parameters())
print(f"✅ BERT model loaded!")
print(f"   Total parameters: {total_params:,}")
print(f"   Model size: {total_params * 4 / 1024 / 1024:.1f} MB")

# ============================================================
# STEP 4: Train BERT Model
# ============================================================
print("\n" + "="*60)
print("STEP 4: TRAINING BERT MODEL")
print("="*60)

# Training arguments
training_args = TrainingArguments(
    output_dir='./bert_results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    eval_strategy='epoch',
    save_strategy='no',
    logging_steps=100,
    load_best_model_at_end=False,
)

# Define metrics function
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    accuracy = accuracy_score(labels, predictions)
    precision = precision_score(labels, predictions, zero_division=0)
    recall = recall_score(labels, predictions, zero_division=0)
    f1 = f1_score(labels, predictions, zero_division=0)
    return {'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1}

# Create trainer
trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    compute_metrics=compute_metrics,
)

# Train
print("\nStarting BERT training (this will take 10-20 minutes)...")
trainer.train()

# ============================================================
# STEP 5: Evaluate BERT Model
# ============================================================
print("\n" + "="*60)
print("STEP 5: EVALUATING BERT MODEL")
print("="*60)

# Get predictions
bert_predictions = trainer.predict(test_tokenized)
bert_pred_probs = np.exp(bert_predictions.predictions)
bert_pred_probs = bert_pred_probs / bert_pred_probs.sum(axis=1, keepdims=True)
bert_pred_labels = np.argmax(bert_predictions.predictions, axis=1)

# Calculate BERT metrics
bert_accuracy = accuracy_score(bert_test_labels, bert_pred_labels)
bert_precision = precision_score(bert_test_labels, bert_pred_labels, zero_division=0)
bert_recall = recall_score(bert_test_labels, bert_pred_labels, zero_division=0)
bert_f1 = f1_score(bert_test_labels, bert_pred_labels, zero_division=0)
bert_auc = roc_auc_score(bert_test_labels, bert_pred_probs[:, 1])

print(f"\n🏆 BERT Results on Test Set:")
print(f"   Accuracy:  {bert_accuracy:.4f}")
print(f"   Precision: {bert_precision:.4f}")
print(f"   Recall:    {bert_recall:.4f}")
print(f"   F1-Score:  {bert_f1:.4f}")
print(f"   AUC:       {bert_auc:.4f}")

# ============================================================
# STEP 6: Evaluate Traditional Models
# ============================================================
print("\n" + "="*60)
print("STEP 6: EVALUATING TRADITIONAL MODELS")
print("="*60)

# Use the existing traditional models from earlier cells
traditional_results = []
traditional_predictions = {}
traditional_probabilities = {}

for model_name in models.keys():
    print(f"\n📊 {model_name}:")
    print("-" * 40)

    try:
        model_obj = models[model_name]

        # Get predictions
        preds = model_obj.predict(X_test)
        if hasattr(preds, 'flatten'):
            preds = preds.flatten()
        else:
            preds = np.array(preds).flatten()

        # Get probabilities
        probs = model_obj.predict_proba(X_test)[:, 1]
        if hasattr(probs, 'flatten'):
            probs = probs.flatten()
        else:
            probs = np.array(probs).flatten()

        # Calculate metrics
        accuracy = accuracy_score(bert_test_labels, preds)
        precision = precision_score(bert_test_labels, preds, zero_division=0)
        recall = recall_score(bert_test_labels, preds, zero_division=0)
        f1 = f1_score(bert_test_labels, preds, zero_division=0)
        auc_score = roc_auc_score(bert_test_labels, probs)

        traditional_results.append({
            'Model': model_name,
            'Accuracy': accuracy,
            'Precision': precision,
            'Recall': recall,
            'F1-Score': f1,
            'AUC': auc_score
        })

        traditional_predictions[model_name] = preds
        traditional_probabilities[model_name] = probs

        print(f"   Accuracy:  {accuracy:.4f}")
        print(f"   Precision: {precision:.4f}")
        print(f"   Recall:    {recall:.4f}")
        print(f"   F1-Score:  {f1:.4f}")
        print(f"   AUC:       {auc_score:.4f}")

    except Exception as e:
        print(f"   ❌ Error: {e}")

# ============================================================
# STEP 7: Combine All Results
# ============================================================
print("\n" + "="*60)
print("STEP 7: COMBINED RESULTS")
print("="*60)

# Add BERT results
traditional_results.append({
    'Model': 'BERT',
    'Accuracy': bert_accuracy,
    'Precision': bert_precision,
    'Recall': bert_recall,
    'F1-Score': bert_f1,
    'AUC': bert_auc
})

# Create DataFrame
results_df = pd.DataFrame(traditional_results)
results_df = results_df.sort_values('Accuracy', ascending=False)

print("\n📊 COMPLETE MODEL COMPARISON:")
print(results_df.round(4).to_string(index=False))

# Save to CSV
results_df.to_csv('complete_model_comparison.csv', index=False)
print("\n✅ Results saved to 'complete_model_comparison.csv'")

# ============================================================
# STEP 8: VISUALIZATIONS
# ============================================================
print("\n" + "="*60)
print("STEP 8: GENERATING VISUALIZATIONS")
print("="*60)

# Create large figure
fig = plt.figure(figsize=(22, 16))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

colors_bar = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6', '#f39c12'][:len(results_df)]

# 8.1 Accuracy Bar Chart
ax1 = fig.add_subplot(gs[0, 0])
bars = ax1.bar(results_df['Model'], results_df['Accuracy'], color=colors_bar, edgecolor='black', linewidth=1.5)
ax1.set_title('Model Accuracy Comparison', fontsize=14, fontweight='bold')
ax1.set_ylabel('Accuracy', fontsize=12)
ax1.set_ylim(0, 1)
for bar in bars:
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height + 0.01, f'{height:.3f}', ha='center', fontweight='bold')
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')

# 8.2 F1-Score Bar Chart
ax2 = fig.add_subplot(gs[0, 1])
bars = ax2.bar(results_df['Model'], results_df['F1-Score'], color=colors_bar, edgecolor='black', linewidth=1.5)
ax2.set_title('Model F1-Score Comparison', fontsize=14, fontweight='bold')
ax2.set_ylabel('F1-Score', fontsize=12)
ax2.set_ylim(0, 1)
for bar in bars:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height + 0.01, f'{height:.3f}', ha='center', fontweight='bold')
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')

# 8.3 AUC Bar Chart
ax3 = fig.add_subplot(gs[0, 2])
bars = ax3.bar(results_df['Model'], results_df['AUC'], color=colors_bar, edgecolor='black', linewidth=1.5)
ax3.set_title('Model AUC Comparison', fontsize=14, fontweight='bold')
ax3.set_ylabel('AUC Score', fontsize=12)
ax3.set_ylim(0, 1)
for bar in bars:
    height = bar.get_height()
    ax3.text(bar.get_x() + bar.get_width()/2., height + 0.01, f'{height:.3f}', ha='center', fontweight='bold')
plt.setp(ax3.xaxis.get_majorticklabels(), rotation=45, ha='right')

# 8.4 Performance Metrics Heatmap
ax4 = fig.add_subplot(gs[1, 0])
metrics_matrix = results_df[['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC']].values
sns.heatmap(metrics_matrix, annot=True, fmt='.4f', cmap='RdYlGn', ax=ax4,
            xticklabels=['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC'],
            yticklabels=results_df['Model'])
ax4.set_title('Performance Metrics Heatmap', fontsize=14, fontweight='bold')

# 8.5 ROC Curves (All Models)
ax5 = fig.add_subplot(gs[1, 1])
colors_roc = {'Naive Bayes': 'blue', 'Logistic Regression': 'green',
              'Random Forest': 'red', 'XGBoost': 'purple', 'BERT': 'orange'}

# Traditional models ROC
for model_name, probs in traditional_probabilities.items():
    fpr, tpr, _ = roc_curve(bert_test_labels, probs)
    roc_auc = auc(fpr, tpr)
    color = colors_roc.get(model_name, 'gray')
    ax5.plot(fpr, tpr, color=color, lw=2, label=f'{model_name} (AUC={roc_auc:.3f})')

# BERT ROC
fpr, tpr, _ = roc_curve(bert_test_labels, bert_pred_probs[:, 1])
roc_auc = auc(fpr, tpr)
ax5.plot(fpr, tpr, color='orange', lw=2, label=f'BERT (AUC={roc_auc:.3f})')

ax5.plot([0, 1], [0, 1], 'gray', lw=2, linestyle='--', label='Random Classifier')
ax5.set_xlabel('False Positive Rate', fontsize=12)
ax5.set_ylabel('True Positive Rate', fontsize=12)
ax5.set_title('ROC Curves - All Models', fontsize=14, fontweight='bold')
ax5.legend(loc='lower right', fontsize=9)
ax5.grid(True, alpha=0.3)

# 8.6 Model Ranking Table
ax6 = fig.add_subplot(gs[1, 2])
ax6.axis('tight')
ax6.axis('off')

ranking_data = results_df[['Model', 'Accuracy', 'F1-Score', 'AUC']].copy()
ranking_data['Rank'] = range(1, len(ranking_data) + 1)
ranking_data = ranking_data[['Rank', 'Model', 'Accuracy', 'F1-Score', 'AUC']]

col_colours = ['#3498db'] * len(ranking_data.columns)
table = ax6.table(cellText=ranking_data.round(4).values,
                   colLabels=ranking_data.columns,
                   cellLoc='center', loc='center',
                   colColours=col_colours)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.5)
ax6.set_title('Model Rankings', fontsize=14, fontweight='bold', pad=20)

# 8.7 Precision vs Recall
ax7 = fig.add_subplot(gs[2, 0])
x = np.arange(len(results_df['Model']))
width = 0.35
ax7.bar(x - width/2, results_df['Precision'], width, label='Precision', color='#2ecc71', edgecolor='black')
ax7.bar(x + width/2, results_df['Recall'], width, label='Recall', color='#e74c3c', edgecolor='black')
ax7.set_xlabel('Model', fontsize=12)
ax7.set_ylabel('Score', fontsize=12)
ax7.set_title('Precision vs Recall Comparison', fontsize=14, fontweight='bold')
ax7.set_xticks(x)
ax7.set_xticklabels(results_df['Model'], rotation=45, ha='right')
ax7.set_ylim(0, 1)
ax7.legend()
ax7.grid(axis='y', alpha=0.3)

# 8.8 Performance Summary
ax8 = fig.add_subplot(gs[2, 1])
ax8.axis('tight')
ax8.axis('off')

best_model_name = results_df.iloc[0]['Model']
best_acc = results_df.iloc[0]['Accuracy']
best_f1 = results_df.iloc[0]['F1-Score']
improvement = (best_acc - results_df.iloc[-1]['Accuracy']) * 100

summary_text = f"""
PERFORMANCE SUMMARY
{'='*30}
Total Models Compared: {len(results_df)}

🏆 BEST MODEL: {best_model_name}
   • Accuracy: {best_acc:.4f} ({best_acc*100:.1f}%)
   • F1-Score: {best_f1:.4f}
   • Improvement: +{improvement:.1f}% over worst

📊 BERT PERFORMANCE
   • Accuracy: {bert_accuracy:.4f}
   • F1-Score: {bert_f1:.4f}
   • AUC: {bert_auc:.4f}
"""
ax8.text(0.1, 0.5, summary_text, transform=ax8.transAxes, fontsize=11,
         verticalalignment='center', fontfamily='monospace')
ax8.set_title('Summary', fontsize=14, fontweight='bold')

# 8.9 Improvement Chart
ax9 = fig.add_subplot(gs[2, 2])
improvements = (results_df['Accuracy'].values - results_df['Accuracy'].values[-1]) * 100
ax9.bar(results_df['Model'], improvements, color='#9b59b6', edgecolor='black')
ax9.set_xlabel('Model', fontsize=12)
ax9.set_ylabel('Improvement over Worst Model (%)', fontsize=12)
ax9.set_title('Performance Improvement vs Baseline', fontsize=14, fontweight='bold')
plt.setp(ax9.xaxis.get_majorticklabels(), rotation=45, ha='right')
ax9.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax9.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('complete_model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Visualization saved as 'complete_model_comparison.png'")

# ============================================================
# STEP 9: Confusion Matrices for All Models
# ============================================================
print("\n" + "="*60)
print("STEP 9: CONFUSION MATRICES")
print("="*60)

# Count models
num_models = len(traditional_predictions) + 1  # +1 for BERT
cols = min(3, num_models)
rows = (num_models + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 4*rows))
if num_models == 1:
    axes = [axes]
else:
    axes = axes.flatten()

idx = 0

# Traditional models confusion matrices
for model_name, preds in traditional_predictions.items():
    cm = confusion_matrix(bert_test_labels, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=['REAL', 'FAKE'], yticklabels=['REAL', 'FAKE'])
    acc = accuracy_score(bert_test_labels, preds)
    axes[idx].set_title(f'{model_name}\nAccuracy: {acc:.4f}', fontsize=11, fontweight='bold')
    axes[idx].set_xlabel('Predicted', fontsize=9)
    axes[idx].set_ylabel('Actual', fontsize=9)
    idx += 1

# BERT confusion matrix
cm = confusion_matrix(bert_test_labels, bert_pred_labels)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
            xticklabels=['REAL', 'FAKE'], yticklabels=['REAL', 'FAKE'])
axes[idx].set_title(f'BERT\nAccuracy: {bert_accuracy:.4f}', fontsize=11, fontweight='bold')
axes[idx].set_xlabel('Predicted', fontsize=9)
axes[idx].set_ylabel('Actual', fontsize=9)
idx += 1

# Hide unused subplots
for j in range(idx, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.savefig('complete_confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Confusion matrices saved as 'complete_confusion_matrices.png'")

# ============================================================
# STEP 10: Final Summary
# ============================================================
print("\n" + "="*80)
print("FINAL SUMMARY")
print("="*80)

print("\n📊 MODEL RANKINGS:")
for i, row in results_df.iterrows():
    print(f"   {i+1}. {row['Model']}: Accuracy={row['Accuracy']:.4f}, F1={row['F1-Score']:.4f}, AUC={row['AUC']:.4f}")

print(f"\n🏆 BEST MODEL: {results_df.iloc[0]['Model']}")
print(f"   Accuracy: {results_df.iloc[0]['Accuracy']:.4f}")
print(f"   F1-Score: {results_df.iloc[0]['F1-Score']:.4f}")

if results_df.iloc[0]['Model'] == 'BERT':
    print("\n🎉 BERT achieved the best performance!")
    print("   Transformers capture linguistic patterns better than traditional ML models.")
    improvement_vs_rf = (bert_accuracy - results_df[results_df['Model'] == 'Random Forest']['Accuracy'].values[0]) * 100
    print(f"   BERT improvement over Random Forest: +{improvement_vs_rf:.1f}%")

# Save final report
with open('final_complete_report.txt', 'w') as f:
    f.write("="*80 + "\n")
    f.write("FAKE NEWS DETECTION - COMPLETE REPORT\n")
    f.write("="*80 + "\n\n")
    f.write(results_df.round(4).to_string(index=False))
    f.write("\n\n" + "="*80 + "\n")
    f.write(f"BEST MODEL: {results_df.iloc[0]['Model']}\n")
    f.write(f"Best Accuracy: {results_df.iloc[0]['Accuracy']:.4f}\n")
    f.write(f"Best F1-Score: {results_df.iloc[0]['F1-Score']:.4f}\n")

print("\n✅ Final report saved to 'final_complete_report.txt'")
print("\n" + "="*80)
print("✅ COMPLETE! BERT trained and compared with all traditional models!")
print("="*80)

In [ ]:
# Cell: Complete RoBERTa Training + Comparison with BERT and Traditional Models
print("\n" + "="*80)
print("COMPLETE FAKE NEWS DETECTION: RoBERTa + BERT + TRADITIONAL MODELS")
print("="*80)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve, auc,
    classification_report
)
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, RobertaTokenizer, RobertaForSequenceClassification
)
from datasets import Dataset
import torch
import warnings
warnings.filterwarnings('ignore')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n🔧 Using device: {device}")
if device.type == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ============================================================
# STEP 1: Prepare Data for RoBERTa and BERT
# ============================================================
print("\n" + "="*60)
print("STEP 1: PREPARING DATA")
print("="*60)

# Get text data with metadata (if available)
def combine_with_metadata(row):
    """Combine statement with speaker, party, and subject information"""
    statement = str(row['statement'])
    speaker = str(row['speaker']) if pd.notna(row['speaker']) and row['speaker'] != '' else 'unknown'
    party = str(row['party']) if pd.notna(row['party']) and row['party'] != '' else 'unknown'
    subject = str(row['subject']) if pd.notna(row['subject']) and row['subject'] != '' else 'unknown'

    return f"[SPEAKER] {speaker} [PARTY] {party} [SUBJECT] {subject} [TEXT] {statement}"

# Check if metadata columns exist
has_metadata = 'speaker' in train_df.columns and 'party' in train_df.columns

if has_metadata:
    print("✅ Metadata columns found (speaker, party, subject)")
    print("   Adding metadata to training and test statements...")
    train_df['enhanced_text'] = train_df.apply(combine_with_metadata, axis=1)
    test_df['enhanced_text'] = test_df.apply(combine_with_metadata, axis=1)

    roberta_train_texts = train_df['enhanced_text'].tolist()
    roberta_test_texts = test_df['enhanced_text'].tolist()
    bert_train_texts = train_df['enhanced_text'].tolist()
    bert_test_texts = test_df['enhanced_text'].tolist()

    print("   📝 Example enhanced text:")
    print(f"      {train_df['enhanced_text'].iloc[0][:150]}...")
else:
    print("⚠️ No metadata columns found. Using statements only.")
    roberta_train_texts = train_df['statement'].tolist()
    roberta_test_texts = test_df['statement'].tolist()
    bert_train_texts = train_df['statement'].tolist()
    bert_test_texts = test_df['statement'].tolist()

# Get labels
if hasattr(y_train, 'values'):
    train_labels = y_train.values.flatten()
    test_labels = y_test.values.flatten()
else:
    train_labels = np.array(y_train).flatten()
    test_labels = np.array(y_test).flatten()

print(f"\n📊 Data Statistics:")
print(f"   Training samples: {len(roberta_train_texts)}")
print(f"   Test samples: {len(roberta_test_texts)}")
print(f"   Train - REAL: {np.sum(train_labels == 0)}, FAKE: {np.sum(train_labels == 1)}")
print(f"   Test - REAL: {np.sum(test_labels == 0)}, FAKE: {np.sum(test_labels == 1)}")

# ============================================================
# STEP 2: Train RoBERTa Model
# ============================================================
print("\n" + "="*60)
print("STEP 2: TRAINING ROBERTA MODEL")
print("="*60)

# Tokenize for RoBERTa
print("Loading RoBERTa tokenizer...")
roberta_tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

def tokenize_for_roberta(examples):
    return roberta_tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=128
    )

# Create datasets
roberta_train_dataset = Dataset.from_dict({'text': roberta_train_texts, 'label': train_labels})
roberta_test_dataset = Dataset.from_dict({'text': roberta_test_texts, 'label': test_labels})

# Tokenize
roberta_train_tokenized = roberta_train_dataset.map(tokenize_for_roberta, batched=True)
roberta_test_tokenized = roberta_test_dataset.map(tokenize_for_roberta, batched=True)

# Remove text column and set format
roberta_train_tokenized = roberta_train_tokenized.remove_columns(['text'])
roberta_test_tokenized = roberta_test_tokenized.remove_columns(['text'])
roberta_train_tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
roberta_test_tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

print(f"✅ RoBERTa tokenization complete!")
print(f"   Train dataset: {len(roberta_train_tokenized)}")
print(f"   Test dataset: {len(roberta_test_tokenized)}")

# Load RoBERTa model
print("\nLoading RoBERTa model...")
roberta_model = RobertaForSequenceClassification.from_pretrained('roberta-base', num_labels=2)
roberta_model = roberta_model.to(device)

roberta_params = sum(p.numel() for p in roberta_model.parameters())
print(f"✅ RoBERTa model loaded!")
print(f"   Total parameters: {roberta_params:,}")
print(f"   Model size: {roberta_params * 4 / 1024 / 1024:.1f} MB")

# Training arguments for RoBERTa
roberta_training_args = TrainingArguments(
    output_dir='./roberta_results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    eval_strategy='epoch',
    save_strategy='no',
    logging_steps=100,
    learning_rate=2e-5,
)


# Define metrics function
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    accuracy = accuracy_score(labels, predictions)
    precision = precision_score(labels, predictions, zero_division=0)
    recall = recall_score(labels, predictions, zero_division=0)
    f1 = f1_score(labels, predictions, zero_division=0)
    return {'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1}

# Create RoBERTa trainer
roberta_trainer = Trainer(
    model=roberta_model,
    args=roberta_training_args,
    train_dataset=roberta_train_tokenized,
    eval_dataset=roberta_test_tokenized,
    compute_metrics=compute_metrics,
)

# Train RoBERTa
print("\n🚀 Starting RoBERTa training (this will take 15-25 minutes)...")
roberta_trainer.train()

# Evaluate RoBERTa
print("\n" + "="*60)
print("STEP 3: ROBERTA EVALUATION")
print("="*60)

roberta_predictions = roberta_trainer.predict(roberta_test_tokenized)
roberta_pred_probs = np.exp(roberta_predictions.predictions)
roberta_pred_probs = roberta_pred_probs / roberta_pred_probs.sum(axis=1, keepdims=True)
roberta_pred_labels = np.argmax(roberta_predictions.predictions, axis=1)

# Calculate RoBERTa metrics
roberta_accuracy = accuracy_score(test_labels, roberta_pred_labels)
roberta_precision = precision_score(test_labels, roberta_pred_labels, zero_division=0)
roberta_recall = recall_score(test_labels, roberta_pred_labels, zero_division=0)
roberta_f1 = f1_score(test_labels, roberta_pred_labels, zero_division=0)
roberta_auc = roc_auc_score(test_labels, roberta_pred_probs[:, 1])

print(f"\n🏆 RoBERTa Results on Test Set:")
print(f"   Accuracy:  {roberta_accuracy:.4f}")
print(f"   Precision: {roberta_precision:.4f}")
print(f"   Recall:    {roberta_recall:.4f}")
print(f"   F1-Score:  {roberta_f1:.4f}")
print(f"   AUC:       {roberta_auc:.4f}")

# ============================================================
# STEP 4: Train BERT Model (if not already trained)
# ============================================================
print("\n" + "="*60)
print("STEP 4: TRAINING BERT MODEL")
print("="*60)

# Check if BERT results already exist
if 'bert_accuracy' not in locals():
    print("Training BERT model...")

    # Tokenize for BERT
    bert_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

    def tokenize_for_bert(examples):
        return bert_tokenizer(
            examples['text'],
            padding='max_length',
            truncation=True,
            max_length=128
        )

    bert_train_dataset = Dataset.from_dict({'text': bert_train_texts, 'label': train_labels})
    bert_test_dataset = Dataset.from_dict({'text': bert_test_texts, 'label': test_labels})

    bert_train_tokenized = bert_train_dataset.map(tokenize_for_bert, batched=True)
    bert_test_tokenized = bert_test_dataset.map(tokenize_for_bert, batched=True)

    bert_train_tokenized = bert_train_tokenized.remove_columns(['text'])
    bert_test_tokenized = bert_test_tokenized.remove_columns(['text'])
    bert_train_tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
    bert_test_tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

    # Load BERT model
    bert_model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
    bert_model = bert_model.to(device)

    # Training arguments
    bert_training_args = TrainingArguments(
        output_dir='./bert_results',
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=64,
        warmup_steps=500,
        weight_decay=0.01,
        logging_dir='./logs',
        eval_strategy='epoch',
        save_strategy='no',
        logging_steps=100,
        learning_rate=2e-5,
    )

    bert_trainer = Trainer(
        model=bert_model,
        args=bert_training_args,
        train_dataset=bert_train_tokenized,
        eval_dataset=bert_test_tokenized,
        compute_metrics=compute_metrics,
    )

    print("🚀 Starting BERT training...")
    bert_trainer.train()

    # Evaluate BERT
    bert_predictions = bert_trainer.predict(bert_test_tokenized)
    bert_pred_probs = np.exp(bert_predictions.predictions)
    bert_pred_probs = bert_pred_probs / bert_pred_probs.sum(axis=1, keepdims=True)
    bert_pred_labels = np.argmax(bert_predictions.predictions, axis=1)

    bert_accuracy = accuracy_score(test_labels, bert_pred_labels)
    bert_precision = precision_score(test_labels, bert_pred_labels, zero_division=0)
    bert_recall = recall_score(test_labels, bert_pred_labels, zero_division=0)
    bert_f1 = f1_score(test_labels, bert_pred_labels, zero_division=0)
    bert_auc = roc_auc_score(test_labels, bert_pred_probs[:, 1])

    print(f"\n🏆 BERT Results on Test Set:")
    print(f"   Accuracy:  {bert_accuracy:.4f}")
    print(f"   Precision: {bert_precision:.4f}")
    print(f"   Recall:    {bert_recall:.4f}")
    print(f"   F1-Score:  {bert_f1:.4f}")
    print(f"   AUC:       {bert_auc:.4f}")
else:
    print("✅ Using existing BERT results from memory")

# ============================================================
# STEP 5: Evaluate Traditional Models
# ============================================================
print("\n" + "="*60)
print("STEP 5: EVALUATING TRADITIONAL MODELS")
print("="*60)

traditional_results = []
traditional_predictions = {}
traditional_probabilities = {}

for model_name in models.keys():
    print(f"\n📊 {model_name}:")
    print("-" * 40)

    try:
        model_obj = models[model_name]

        # Get predictions
        preds = model_obj.predict(X_test)
        if hasattr(preds, 'flatten'):
            preds = preds.flatten()
        else:
            preds = np.array(preds).flatten()

        # Get probabilities
        probs = model_obj.predict_proba(X_test)[:, 1]
        if hasattr(probs, 'flatten'):
            probs = probs.flatten()
        else:
            probs = np.array(probs).flatten()

        # Calculate metrics
        accuracy = accuracy_score(test_labels, preds)
        precision = precision_score(test_labels, preds, zero_division=0)
        recall = recall_score(test_labels, preds, zero_division=0)
        f1 = f1_score(test_labels, preds, zero_division=0)
        auc_score = roc_auc_score(test_labels, probs)

        traditional_results.append({
            'Model': model_name,
            'Accuracy': accuracy,
            'Precision': precision,
            'Recall': recall,
            'F1-Score': f1,
            'AUC': auc_score
        })

        traditional_predictions[model_name] = preds
        traditional_probabilities[model_name] = probs

        print(f"   Accuracy:  {accuracy:.4f}")
        print(f"   Precision: {precision:.4f}")
        print(f"   Recall:    {recall:.4f}")
        print(f"   F1-Score:  {f1:.4f}")
        print(f"   AUC:       {auc_score:.4f}")

    except Exception as e:
        print(f"   ❌ Error: {e}")

# ============================================================
# STEP 6: Combine All Results
# ============================================================
print("\n" + "="*60)
print("STEP 6: COMBINED RESULTS - ALL MODELS")
print("="*60)

# Add RoBERTa results
traditional_results.append({
    'Model': 'RoBERTa',
    'Accuracy': roberta_accuracy,
    'Precision': roberta_precision,
    'Recall': roberta_recall,
    'F1-Score': roberta_f1,
    'AUC': roberta_auc
})

# Add BERT results
traditional_results.append({
    'Model': 'BERT',
    'Accuracy': bert_accuracy,
    'Precision': bert_precision,
    'Recall': bert_recall,
    'F1-Score': bert_f1,
    'AUC': bert_auc
})

# Create DataFrame
results_df = pd.DataFrame(traditional_results)
results_df = results_df.sort_values('Accuracy', ascending=False)

print("\n📊 COMPLETE MODEL COMPARISON:")
print(results_df.round(4).to_string(index=False))

# Save to CSV
results_df.to_csv('roberta_bert_comparison.csv', index=False)
print("\n✅ Results saved to 'roberta_bert_comparison.csv'")

# ============================================================
# STEP 7: VISUALIZATIONS
# ============================================================
print("\n" + "="*60)
print("STEP 7: GENERATING VISUALIZATIONS")
print("="*60)

# Create large figure
fig = plt.figure(figsize=(22, 16))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

colors_bar = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6', '#f39c12', '#1abc9c'][:len(results_df)]

# 7.1 Accuracy Bar Chart
ax1 = fig.add_subplot(gs[0, 0])
bars = ax1.bar(results_df['Model'], results_df['Accuracy'], color=colors_bar, edgecolor='black', linewidth=1.5)
ax1.set_title('Model Accuracy Comparison', fontsize=14, fontweight='bold')
ax1.set_ylabel('Accuracy', fontsize=12)
ax1.set_ylim(0, 1)
for bar in bars:
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height + 0.01, f'{height:.3f}', ha='center', fontweight='bold')
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')

# 7.2 F1-Score Bar Chart
ax2 = fig.add_subplot(gs[0, 1])
bars = ax2.bar(results_df['Model'], results_df['F1-Score'], color=colors_bar, edgecolor='black', linewidth=1.5)
ax2.set_title('Model F1-Score Comparison', fontsize=14, fontweight='bold')
ax2.set_ylabel('F1-Score', fontsize=12)
ax2.set_ylim(0, 1)
for bar in bars:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height + 0.01, f'{height:.3f}', ha='center', fontweight='bold')
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')

# 7.3 AUC Bar Chart
ax3 = fig.add_subplot(gs[0, 2])
bars = ax3.bar(results_df['Model'], results_df['AUC'], color=colors_bar, edgecolor='black', linewidth=1.5)
ax3.set_title('Model AUC Comparison', fontsize=14, fontweight='bold')
ax3.set_ylabel('AUC Score', fontsize=12)
ax3.set_ylim(0, 1)
for bar in bars:
    height = bar.get_height()
    ax3.text(bar.get_x() + bar.get_width()/2., height + 0.01, f'{height:.3f}', ha='center', fontweight='bold')
plt.setp(ax3.xaxis.get_majorticklabels(), rotation=45, ha='right')

# 7.4 Performance Metrics Heatmap
ax4 = fig.add_subplot(gs[1, 0])
metrics_matrix = results_df[['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC']].values
sns.heatmap(metrics_matrix, annot=True, fmt='.4f', cmap='RdYlGn', ax=ax4,
            xticklabels=['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC'],
            yticklabels=results_df['Model'])
ax4.set_title('Performance Metrics Heatmap', fontsize=14, fontweight='bold')

# 7.5 ROC Curves (All Models)
ax5 = fig.add_subplot(gs[1, 1])
colors_roc = {'Naive Bayes': 'blue', 'Logistic Regression': 'green',
              'Random Forest': 'red', 'XGBoost': 'purple',
              'BERT': 'orange', 'RoBERTa': 'brown'}

# Traditional models ROC
for model_name, probs in traditional_probabilities.items():
    fpr, tpr, _ = roc_curve(test_labels, probs)
    roc_auc = auc(fpr, tpr)
    color = colors_roc.get(model_name, 'gray')
    ax5.plot(fpr, tpr, color=color, lw=2, label=f'{model_name} (AUC={roc_auc:.3f})')

# BERT ROC
fpr, tpr, _ = roc_curve(test_labels, bert_pred_probs[:, 1])
roc_auc = auc(fpr, tpr)
ax5.plot(fpr, tpr, color='orange', lw=2, label=f'BERT (AUC={roc_auc:.3f})')

# RoBERTa ROC
fpr, tpr, _ = roc_curve(test_labels, roberta_pred_probs[:, 1])
roc_auc = auc(fpr, tpr)
ax5.plot(fpr, tpr, color='brown', lw=2, label=f'RoBERTa (AUC={roc_auc:.3f})')

ax5.plot([0, 1], [0, 1], 'gray', lw=2, linestyle='--', label='Random Classifier')
ax5.set_xlabel('False Positive Rate', fontsize=12)
ax5.set_ylabel('True Positive Rate', fontsize=12)
ax5.set_title('ROC Curves - All Models', fontsize=14, fontweight='bold')
ax5.legend(loc='lower right', fontsize=8)
ax5.grid(True, alpha=0.3)

# 7.6 Model Ranking Table
ax6 = fig.add_subplot(gs[1, 2])
ax6.axis('tight')
ax6.axis('off')

ranking_data = results_df[['Model', 'Accuracy', 'F1-Score', 'AUC']].copy()
ranking_data['Rank'] = range(1, len(ranking_data) + 1)
ranking_data = ranking_data[['Rank', 'Model', 'Accuracy', 'F1-Score', 'AUC']]

col_colours = ['#3498db'] * len(ranking_data.columns)
table = ax6.table(cellText=ranking_data.round(4).values,
                   colLabels=ranking_data.columns,
                   cellLoc='center', loc='center',
                   colColours=col_colours)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.5)
ax6.set_title('Model Rankings', fontsize=14, fontweight='bold', pad=20)

# 7.7 Precision vs Recall
ax7 = fig.add_subplot(gs[2, 0])
x = np.arange(len(results_df['Model']))
width = 0.35
ax7.bar(x - width/2, results_df['Precision'], width, label='Precision', color='#2ecc71', edgecolor='black')
ax7.bar(x + width/2, results_df['Recall'], width, label='Recall', color='#e74c3c', edgecolor='black')
ax7.set_xlabel('Model', fontsize=12)
ax7.set_ylabel('Score', fontsize=12)
ax7.set_title('Precision vs Recall Comparison', fontsize=14, fontweight='bold')
ax7.set_xticks(x)
ax7.set_xticklabels(results_df['Model'], rotation=45, ha='right')
ax7.set_ylim(0, 1)
ax7.legend()
ax7.grid(axis='y', alpha=0.3)

# 7.8 Performance Summary
ax8 = fig.add_subplot(gs[2, 1])
ax8.axis('tight')
ax8.axis('off')

best_model_name = results_df.iloc[0]['Model']
best_acc = results_df.iloc[0]['Accuracy']
best_f1 = results_df.iloc[0]['F1-Score']

# Calculate improvements
roberta_vs_bert = (roberta_accuracy - bert_accuracy) * 100
roberta_vs_rf = (roberta_accuracy - results_df[results_df['Model'] == 'Random Forest']['Accuracy'].values[0]) * 100 if 'Random Forest' in results_df['Model'].values else 0

summary_text = f"""
PERFORMANCE SUMMARY
{'='*30}
Total Models Compared: {len(results_df)}

🏆 BEST MODEL: {best_model_name}
   • Accuracy: {best_acc:.4f} ({best_acc*100:.1f}%)
   • F1-Score: {best_f1:.4f}

📊 ROBERTA vs BERT
   • RoBERTa: {roberta_accuracy:.4f}
   • BERT: {bert_accuracy:.4f}
   • Difference: {roberta_vs_bert:+.1f}%

📈 IMPROVEMENT OVER RANDOM FOREST
   • RoBERTa: +{roberta_vs_rf:.1f}%
"""
ax8.text(0.1, 0.5, summary_text, transform=ax8.transAxes, fontsize=11,
         verticalalignment='center', fontfamily='monospace')
ax8.set_title('Summary', fontsize=14, fontweight='bold')

# 7.9 Improvement Chart
ax9 = fig.add_subplot(gs[2, 2])
improvements = (results_df['Accuracy'].values - results_df['Accuracy'].values[-1]) * 100
colors_improve = ['#2ecc71' if i == 0 else '#3498db' for i in range(len(improvements))]
bars = ax9.bar(results_df['Model'], improvements, color=colors_improve, edgecolor='black')
ax9.set_xlabel('Model', fontsize=12)
ax9.set_ylabel('Improvement over Worst Model (%)', fontsize=12)
ax9.set_title('Performance Improvement vs Baseline', fontsize=14, fontweight='bold')
plt.setp(ax9.xaxis.get_majorticklabels(), rotation=45, ha='right')
ax9.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax9.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar, imp in zip(bars, improvements):
    if imp > 0:
        ax9.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5, f'+{imp:.1f}%', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('roberta_bert_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Visualization saved as 'roberta_bert_comparison.png'")

# ============================================================
# STEP 8: Confusion Matrices for All Models
# ============================================================
print("\n" + "="*60)
print("STEP 8: CONFUSION MATRICES")
print("="*60)

# Count models
num_models = len(traditional_predictions) + 2  # +2 for BERT and RoBERTa
cols = min(3, num_models)
rows = (num_models + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 4*rows))
if num_models == 1:
    axes = [axes]
else:
    axes = axes.flatten()

idx = 0

# Traditional models confusion matrices
for model_name, preds in traditional_predictions.items():
    cm = confusion_matrix(test_labels, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=['REAL', 'FAKE'], yticklabels=['REAL', 'FAKE'])
    acc = accuracy_score(test_labels, preds)
    axes[idx].set_title(f'{model_name}\nAccuracy: {acc:.4f}', fontsize=11, fontweight='bold')
    axes[idx].set_xlabel('Predicted', fontsize=9)
    axes[idx].set_ylabel('Actual', fontsize=9)
    idx += 1

# BERT confusion matrix
cm = confusion_matrix(test_labels, bert_pred_labels)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
            xticklabels=['REAL', 'FAKE'], yticklabels=['REAL', 'FAKE'])
axes[idx].set_title(f'BERT\nAccuracy: {bert_accuracy:.4f}', fontsize=11, fontweight='bold')
axes[idx].set_xlabel('Predicted', fontsize=9)
axes[idx].set_ylabel('Actual', fontsize=9)
idx += 1

# RoBERTa confusion matrix
cm = confusion_matrix(test_labels, roberta_pred_labels)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
            xticklabels=['REAL', 'FAKE'], yticklabels=['REAL', 'FAKE'])
axes[idx].set_title(f'RoBERTa\nAccuracy: {roberta_accuracy:.4f}', fontsize=11, fontweight='bold')
axes[idx].set_xlabel('Predicted', fontsize=9)
axes[idx].set_ylabel('Actual', fontsize=9)
idx += 1

# Hide unused subplots
for j in range(idx, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.savefig('roberta_bert_confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Confusion matrices saved as 'roberta_bert_confusion_matrices.png'")

# ============================================================
# STEP 9: Final Summary
# ============================================================
print("\n" + "="*80)
print("FINAL SUMMARY - ALL MODELS")
print("="*80)

print("\n📊 MODEL RANKINGS (by Accuracy):")
for i, row in results_df.iterrows():
    print(f"   {i+1}. {row['Model']}: {row['Accuracy']:.4f} (F1: {row['F1-Score']:.4f}, AUC: {row['AUC']:.4f})")

print(f"\n🏆 BEST MODEL: {results_df.iloc[0]['Model']}")
print(f"   Accuracy: {results_df.iloc[0]['Accuracy']:.4f}")
print(f"   F1-Score: {results_df.iloc[0]['F1-Score']:.4f}")

print(f"\n📊 MODEL COMPARISON:")
print(f"   RoBERTa vs BERT: {roberta_accuracy:.4f} vs {bert_accuracy:.4f} (difference: {(roberta_accuracy - bert_accuracy)*100:+.1f}%)")

# Find Random Forest accuracy
rf_row = results_df[results_df['Model'] == 'Random Forest']
if len(rf_row) > 0:
    rf_acc = rf_row['Accuracy'].values[0]
    print(f"\n📈 IMPROVEMENT OVER RANDOM FOREST:")
    print(f"   RoBERTa: +{(roberta_accuracy - rf_acc)*100:.1f}%")
    print(f"   BERT: +{(bert_accuracy - rf_acc)*100:.1f}%")

# Save final report
with open('roberta_bert_final_report.txt', 'w') as f:
    f.write("="*80 + "\n")
    f.write("FAKE NEWS DETECTION - COMPLETE REPORT (RoBERTa + BERT)\n")
    f.write("="*80 + "\n\n")
    f.write(results_df.round(4).to_string(index=False))
    f.write("\n\n" + "="*80 + "\n")
    f.write(f"BEST MODEL: {results_df.iloc[0]['Model']}\n")
    f.write(f"Best Accuracy: {results_df.iloc[0]['Accuracy']:.4f}\n")
    f.write(f"Best F1-Score: {results_df.iloc[0]['F1-Score']:.4f}\n")
    f.write(f"\nRoBERTa vs BERT: {roberta_accuracy:.4f} vs {bert_accuracy:.4f}\n")

print("\n✅ Final report saved to 'roberta_bert_final_report.txt'")

print("\n" + "="*80)
print("✅ COMPLETE! RoBERTa trained and compared with BERT and traditional models!")
print("="*80)

In [ ]:
# Cell: LIME and SHAP Explanations for All Models (COMPLETELY FIXED)
print("\n" + "="*80)
print("LIME AND SHAP EXPLANATIONS FOR ALL MODELS")
print("="*80)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import warnings
import os
warnings.filterwarnings('ignore')

# Install required libraries if not already installed
!pip install lime shap -q

import lime
from lime.lime_text import LimeTextExplainer
import shap

# ============================================================
# STEP 1: Load All Models from Your Training Cell
# ============================================================
print("\n" + "="*60)
print("STEP 1: LOADING ALL MODELS FROM TRAINING")
print("="*60)

# Dictionary to store models
all_models_for_explanation = {}
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔧 Using device: {device}")

# 1.1 Traditional Models (from your 'models' dictionary)
print("\n📚 Loading Traditional Models...")
if 'models' in globals() and models:
    all_models_for_explanation.update(models)
    print(f"✅ Traditional models loaded: {list(models.keys())}")
else:
    print("⚠️ No traditional models found in 'models' dictionary")

# 1.2 BERT Model and Tokenizer (FIXED)
print("\n🤗 Loading BERT Model...")
if 'bert_model' in globals() and bert_model is not None:
    all_models_for_explanation['BERT'] = bert_model
    print(f"✅ BERT model loaded from training")

    # FIX: Check for tokenizer (it's named 'tokenizer' in your training cell)
    if 'tokenizer' in globals() and tokenizer is not None:
        bert_tokenizer = tokenizer  # Create alias
        print(f"✅ BERT tokenizer loaded from 'tokenizer' variable")
    elif 'bert_tokenizer' in globals() and bert_tokenizer is not None:
        print(f"✅ BERT tokenizer available")
    else:
        print(f"⚠️ BERT tokenizer not found - BERT LIME will be skipped")
else:
    print("⚠️ BERT model not found. Make sure you ran the training cell first.")

# 1.3 RoBERTa Model and Tokenizer
print("\n🦝 Loading RoBERTa Model...")
if 'roberta_model' in globals() and roberta_model is not None:
    all_models_for_explanation['RoBERTa'] = roberta_model
    print(f"✅ RoBERTa model loaded from training")

    if 'roberta_tokenizer' in globals() and roberta_tokenizer is not None:
        print(f"✅ RoBERTa tokenizer available")
    else:
        print(f"⚠️ RoBERTa tokenizer not found - RoBERTa LIME may fail")
else:
    print("⚠️ RoBERTa model not found. Make sure you ran the training cell first.")

print(f"\n📊 Total models loaded: {len(all_models_for_explanation)}")
print(f"   Models: {list(all_models_for_explanation.keys())}")

# ============================================================
# STEP 2: Load Data and Vectorizers
# ============================================================
print("\n" + "="*60)
print("STEP 2: PREPARING DATA FOR EXPLANATIONS")
print("="*60)

# Get TF-IDF vectorizer (for traditional models)
tfidf_vectorizer = None
if 'tfidf_vectorizer' in globals():
    tfidf_vectorizer = tfidf_vectorizer
    print("✅ TF-IDF vectorizer found")
elif 'tfidf' in globals():
    tfidf_vectorizer = tfidf
    print("✅ TF-IDF vectorizer found (as 'tfidf')")
else:
    print("⚠️ TF-IDF vectorizer not found - Traditional models will be skipped")

# Get test data with original texts
test_texts = None
test_labels = None

# Try to get test_df from training cell
if 'test_df' in globals():
    if 'statement' in test_df.columns:
        test_texts = test_df['statement'].tolist()
        print("✅ Test texts loaded from test_df['statement']")
    if 'binary_label' in test_df.columns:
        test_labels = test_df['binary_label'].values
        print("✅ Test labels loaded from test_df['binary_label']")

# If still no data, use training data as fallback
if test_texts is None or len(test_texts) == 0:
    if 'train_df' in globals():
        test_texts = train_df['statement'].tolist()[:100]
        if 'binary_label' in train_df.columns:
            test_labels = train_df['binary_label'].values[:100]
        print(f"⚠️ Using first 100 training samples as fallback")

if test_texts is not None and len(test_texts) > 0:
    if test_labels is not None:
        test_labels = np.array(test_labels).flatten()
        print(f"\n📊 Data statistics:")
        print(f"   Total samples: {len(test_texts)}")
        print(f"   REAL: {np.sum(test_labels == 0)}, FAKE: {np.sum(test_labels == 1)}")
    else:
        print(f"\n📊 Data statistics:")
        print(f"   Total samples: {len(test_texts)}")
        print(f"   ⚠️ No labels found")
else:
    print("❌ No test data available. Please run training cell first.")
    raise SystemExit("Cannot proceed without data")

# ============================================================
# STEP 3: Select Diverse Examples for Explanation
# ============================================================
print("\n" + "="*60)
print("STEP 3: SELECTING EXAMPLES FOR EXPLANATION")
print("="*60)

# Simple selection: one REAL, one FAKE, one random
example_indices = []
if test_labels is not None:
    real_idx = np.where(test_labels == 0)[0]
    fake_idx = np.where(test_labels == 1)[0]
    if len(real_idx) > 0:
        example_indices.append(real_idx[0])
    if len(fake_idx) > 0:
        example_indices.append(fake_idx[0])
    if len(example_indices) < 3 and len(test_labels) > 2:
        example_indices.append(2)
else:
    example_indices = [0, 1, 2] if len(test_texts) >= 3 else list(range(len(test_texts)))

selected_texts = [test_texts[i] for i in example_indices[:3]]
selected_labels = [test_labels[i] if test_labels is not None else None for i in example_indices[:3]]
selected_labels_text = ['FAKE' if label == 1 else 'REAL' if label == 0 else 'UNKNOWN' for label in selected_labels]

print(f"\n📝 Selected {len(selected_texts)} examples for explanation:")
for i, (text, label) in enumerate(zip(selected_texts, selected_labels_text)):
    print(f"\n   Example {i+1}: {label}")
    print(f"   Text: {text[:150]}...")

# ============================================================
# STEP 4: Helper Functions for Predictions
# ============================================================
print("\n" + "="*60)
print("STEP 4: SETTING UP PREDICTION FUNCTIONS")
print("="*60)

# Function for traditional models
def predict_proba_traditional(texts, model):
    """Predict probabilities for traditional models using TF-IDF"""
    if tfidf_vectorizer is None:
        raise ValueError("TF-IDF vectorizer not available")
    text_features = tfidf_vectorizer.transform(texts)
    return model.predict_proba(text_features)

# Function for BERT (FIXED - uses 'tokenizer' variable)
def predict_proba_bert(texts):
    """Predict probabilities for BERT model"""
    # Check both possible tokenizer variable names
    if 'tokenizer' in globals() and tokenizer is not None:
        bert_tokenizer_local = tokenizer
    elif 'bert_tokenizer' in globals() and bert_tokenizer is not None:
        bert_tokenizer_local = bert_tokenizer
    else:
        raise ValueError("BERT tokenizer not available")

    inputs = bert_tokenizer_local(texts, padding=True, truncation=True, max_length=128, return_tensors='pt')
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = bert_model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)

    return probs.cpu().numpy()

# Function for RoBERTa
def predict_proba_roberta(texts):
    """Predict probabilities for RoBERTa model"""
    if 'roberta_tokenizer' not in globals() or roberta_tokenizer is None:
        raise ValueError("RoBERTa tokenizer not available")

    inputs = roberta_tokenizer(texts, padding=True, truncation=True, max_length=128, return_tensors='pt')
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = roberta_model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)

    return probs.cpu().numpy()

print("✅ Prediction functions ready")

# ============================================================
# STEP 5: LIME Explanations for All Models
# ============================================================
print("\n" + "="*60)
print("STEP 5: GENERATING LIME EXPLANATIONS")
print("="*60)

lime_explainer = LimeTextExplainer(class_names=['REAL', 'FAKE'])
lime_results = {}

# 5.1 Traditional Models
if tfidf_vectorizer is not None:
    for model_name, model in all_models_for_explanation.items():
        # Skip transformer models for now
        if model_name in ['BERT', 'RoBERTa']:
            continue

        print(f"\n{'─'*50}")
        print(f"🔍 LIME for {model_name}")
        print(f"{'─'*50}")

        model_results = []
        for i, (text, true_label) in enumerate(zip(selected_texts, selected_labels_text)):
            try:
                # Get prediction
                text_features = tfidf_vectorizer.transform([text])
                pred_proba = model.predict_proba(text_features)[0]
                pred_class = "FAKE" if pred_proba[1] > 0.5 else "REAL"
                confidence = max(pred_proba) * 100

                print(f"\n📝 Example {i+1} (True: {true_label})")
                print(f"   Prediction: {pred_class} (Confidence: {confidence:.1f}%)")
                print(f"   Text: {text[:80]}...")

                exp = lime_explainer.explain_instance(
                    text,
                    lambda texts: predict_proba_traditional(texts, model),
                    num_features=6,
                    labels=[1]
                )

                explanation_list = exp.as_list(label=1)
                print(f"\n   🔍 Top words influencing prediction:")
                for word, weight in explanation_list[:5]:
                    direction = "→ FAKE" if weight > 0 else "→ REAL"
                    bar = "█" * min(int(abs(weight) * 20), 15)
                    print(f"      '{word}': {weight:+.4f} {direction} {bar}")

                model_results.append({
                    'example': i+1,
                    'true_label': true_label,
                    'prediction': pred_class,
                    'confidence': confidence,
                    'top_words': [w for w, _ in explanation_list[:3]],
                    'top_weights': [w for _, w in explanation_list[:3]]
                })
            except Exception as e:
                print(f"   ⚠️ Error: {e}")

        if model_results:
            lime_results[model_name] = model_results
else:
    print("\n⚠️ No TF-IDF vectorizer - skipping traditional models")

# 5.2 BERT Model
if 'BERT' in all_models_for_explanation and ('tokenizer' in globals() or 'bert_tokenizer' in globals()):
    print(f"\n{'─'*50}")
    print(f"🔍 LIME for BERT")
    print(f"{'─'*50}")

    bert_model.eval()
    bert_results = []

    for i, (text, true_label) in enumerate(zip(selected_texts, selected_labels_text)):
        try:
            probs = predict_proba_bert([text])[0]
            pred_class = "FAKE" if probs[1] > 0.5 else "REAL"
            confidence = max(probs) * 100

            print(f"\n📝 Example {i+1} (True: {true_label})")
            print(f"   Prediction: {pred_class} (Confidence: {confidence:.1f}%)")
            print(f"   Text: {text[:80]}...")

            exp = lime_explainer.explain_instance(
                text,
                predict_proba_bert,
                num_features=6,
                labels=[1]
            )

            explanation_list = exp.as_list(label=1)
            print(f"\n   🔍 Top words influencing prediction:")
            for word, weight in explanation_list[:5]:
                direction = "→ FAKE" if weight > 0 else "→ REAL"
                bar = "█" * min(int(abs(weight) * 20), 15)
                print(f"      '{word}': {weight:+.4f} {direction} {bar}")

            bert_results.append({
                'example': i+1,
                'true_label': true_label,
                'prediction': pred_class,
                'confidence': confidence,
                'top_words': [w for w, _ in explanation_list[:3]],
                'top_weights': [w for _, w in explanation_list[:3]]
            })
        except Exception as e:
            print(f"   ⚠️ Error for BERT: {e}")

    if bert_results:
        lime_results['BERT'] = bert_results
else:
    print("\n⚠️ BERT not available for LIME (missing model or tokenizer)")

# 5.3 RoBERTa Model
if 'RoBERTa' in all_models_for_explanation and 'roberta_tokenizer' in globals():
    print(f"\n{'─'*50}")
    print(f"🔍 LIME for RoBERTa")
    print(f"{'─'*50}")

    roberta_model.eval()
    roberta_results = []

    for i, (text, true_label) in enumerate(zip(selected_texts, selected_labels_text)):
        try:
            probs = predict_proba_roberta([text])[0]
            pred_class = "FAKE" if probs[1] > 0.5 else "REAL"
            confidence = max(probs) * 100

            print(f"\n📝 Example {i+1} (True: {true_label})")
            print(f"   Prediction: {pred_class} (Confidence: {confidence:.1f}%)")
            print(f"   Text: {text[:80]}...")

            exp = lime_explainer.explain_instance(
                text,
                predict_proba_roberta,
                num_features=6,
                labels=[1]
            )

            explanation_list = exp.as_list(label=1)
            print(f"\n   🔍 Top words influencing prediction:")
            for word, weight in explanation_list[:5]:
                direction = "→ FAKE" if weight > 0 else "→ REAL"
                bar = "█" * min(int(abs(weight) * 20), 15)
                print(f"      '{word}': {weight:+.4f} {direction} {bar}")

            roberta_results.append({
                'example': i+1,
                'true_label': true_label,
                'prediction': pred_class,
                'confidence': confidence,
                'top_words': [w for w, _ in explanation_list[:3]],
                'top_weights': [w for _, w in explanation_list[:3]]
            })
        except Exception as e:
            print(f"   ⚠️ Error for RoBERTa: {e}")

    if roberta_results:
        lime_results['RoBERTa'] = roberta_results
else:
    print("\n⚠️ RoBERTa not available for LIME")

# ============================================================
# STEP 6: Results Summary
# ============================================================
print("\n" + "="*60)
print("STEP 6: RESULTS SUMMARY")
print("="*60)

if lime_results:
    print("\n✅ LIME explanations generated for:")
    for model_name in lime_results.keys():
        print(f"   • {model_name} ({len(lime_results[model_name])} examples)")

    # Create summary table
    summary_data = []
    for model_name, results in lime_results.items():
        for r in results:
            summary_data.append({
                'Model': model_name,
                'Example': f"Example {r['example']}",
                'True Label': r['true_label'],
                'Prediction': r['prediction'],
                'Confidence': f"{r['confidence']:.1f}%",
                'Top 3 Words': ', '.join(r['top_words'])
            })

    summary_df = pd.DataFrame(summary_data)
    print("\n📊 LIME Summary Table:")
    print(summary_df.to_string(index=False))

    # Save to CSV
    summary_df.to_csv('lime_explanations_complete.csv', index=False)
    print("\n✅ Saved to 'lime_explanations_complete.csv'")

    # Visualization
    fig, ax = plt.subplots(figsize=(12, 6))
    models_list = []
    confidences_list = []
    colors_list = []

    for model_name, results in lime_results.items():
        for r in results:
            models_list.append(f"{model_name}\nEx{r['example']}")
            confidences_list.append(r['confidence'])
            colors_list.append('green' if r['prediction'] == r['true_label'] else 'red')

    bars = ax.bar(models_list, confidences_list, color=colors_list, edgecolor='black', linewidth=1.5)
    ax.set_ylim(0, 100)
    ax.set_ylabel('Confidence (%)', fontsize=12)
    ax.set_title('Model Predictions Confidence - LIME Analysis', fontsize=14, fontweight='bold')
    ax.axhline(y=50, color='gray', linestyle='--', linewidth=1, label='Decision Boundary')

    for bar, conf in zip(bars, confidences_list):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 2,
               f'{conf:.0f}%', ha='center', fontsize=9, fontweight='bold')

    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig('lime_all_models_complete.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("✅ Visualization saved to 'lime_all_models_complete.png'")

else:
    print("❌ No LIME explanations were generated")
    print("\nDebugging suggestions:")
    print("1. Make sure you ran ALL training cells first")
    print("2. Check that models are properly trained")
    print("3. Verify that tokenizers are available")
    print("\nCurrent available models:", list(all_models_for_explanation.keys()))
    print("TF-IDF available:", tfidf_vectorizer is not None)
    print("BERT tokenizer available:", 'tokenizer' in globals() or 'bert_tokenizer' in globals())
    print("RoBERTa tokenizer available:", 'roberta_tokenizer' in globals())

print("\n" + "="*80)
print("✅ EXPLANATION ANALYSIS COMPLETE!")
print("="*80)